# 03. 에이전트 행동 경계 설계

## 목표
K3의 명시된 '과도한 주도성' 위험을 줄이기 위한 간단한 정책 검사기를 구현합니다. 실제 운영에서는 권한 샌드박스와 사람의 승인을 함께 사용해야 합니다.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class Decision(Enum):
    ALLOW = "allow"
    ASK = "ask_user"
    DENY = "deny"

@dataclass
class Action:
    name: str
    destructive: bool = False
    external_side_effect: bool = False
    target_is_explicit: bool = True

def evaluate(action: Action) -> Decision:
    # 파괴적이고 대상도 불명확한 행동은 확인만으로 부족하므로 거부합니다.
    if action.destructive and not action.target_is_explicit:
        return Decision.DENY
    # 외부 전송이나 명확한 파괴 작업은 실행 전에 사람에게 확인합니다.
    if action.destructive or action.external_side_effect:
        return Decision.ASK
    return Decision.ALLOW

In [ ]:
actions = [
    Action("테스트 실행"),
    Action("고객에게 이메일 전송", external_side_effect=True),
    Action("명시된 임시 파일 삭제", destructive=True),
    Action("알 수 없는 경로 재귀 삭제", destructive=True, target_is_explicit=False),
]
for action in actions:
    print(f"{action.name}: {evaluate(action).value}")

## Preserved thinking message 계약 검사
K3 다중 턴 API는 이전 assistant message의 `content`, `reasoning_content`, `tool_calls`를 반환된 형태 그대로 다시 전달해야 합니다. 아래 검사는 네트워크 호출 없이 필수 사고 이력 field 누락을 찾는 toy validator입니다.

In [ ]:
def validate_preserved_thinking(messages: list[dict]) -> list[str]:
    errors = []
    for index, message in enumerate(messages):
        if message.get("role") != "assistant":
            continue
        if "reasoning_content" not in message:
            errors.append(f"messages[{index}]: reasoning_content 누락")
        if message.get("tool_calls") and "content" not in message:
            errors.append(f"messages[{index}]: tool call과 함께 content field 누락")
    return errors

conversation = [
    {"role": "user", "content": "두 수를 골라 주세요."},
    {
        "role": "assistant",
        "reasoning_content": "후보 3, 7, 11에서 앞의 두 수를 고른다.",
        "content": "3과 7입니다.",
        "tool_calls": [],
    },
    {"role": "user", "content": "남은 수는 무엇인가요?"},
]
print(validate_preserved_thinking(conversation) or "계약 검사 통과")

## 확장 과제
1. 비용 한도, 실행 시간, 허용 디렉터리와 네트워크 도메인 규칙을 추가하세요.
2. `ASK` 결과에 사용자에게 보여줄 정확한 대상과 예상 영향을 포함하세요.
3. 행동 결정과 결과를 append-only 감사 로그로 남기세요.
4. assistant message를 임의로 재구성하지 않도록 API 응답 hash와 schema version을 기록하세요.
5. `trust_remote_code=True` 사용 전 허용 revision SHA와 code review 결과를 검사하세요.

핵심 원칙은 모델의 말이 아니라 실행 계층에서 권한과 상태 계약을 강제하는 것입니다.